<a target="_blank" href="https://colab.research.google.com/github/TransformerLensOrg/TransformerLens/blob/main/demos/ICA_Lens_Demo.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICA Lens Demo

An **ICA Lens** maps a residual-stream activation into independent-component coordinates. This minimal example captures Gemma 2 2B activations with TransformerLens, decomposes them, and reconstructs the original vectors.

## 1. Load Gemma 2 2B and its published ICA Lens

The fitted matrices are hosted separately on the Hugging Face Hub. `ICALens` does not own the language model or capture activations itself.

In [1]:
# NBVAL_IGNORE_OUTPUT
import torch

from transformer_lens import TransformerBridge
from transformer_lens.tools.analysis import ICALens, JacobianLens, logit_readout

MODEL_ID = "google/gemma-2-2b"
MODEL_REVISION = "c5ebcd40d208330abc697524c919956e692655cf"
device = "cuda" if torch.cuda.is_available() else "cpu"
model_dtype = torch.bfloat16 if device == "cpu" or torch.cuda.is_bf16_supported() else torch.float16
model = TransformerBridge.boot_transformers(
    MODEL_ID, revision=MODEL_REVISION, dtype=model_dtype, device=device
)
model.eval()
ica_lens = ICALens.from_pretrained("gemma-2-2b")
j_lens = JacobianLens.from_pretrained("gemma-2-2b", model=model)
ica_lens

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

ICALens(layers=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25], d_model=2304, preprocessing='none')

## 2. Capture and decompose a residual vector

`run_with_cache` returns the ordinary TransformerLens activation cache. The lens accepts any floating-point tensor whose final dimension is the fitted model width, so batch and token dimensions are preserved.

In [2]:
layer = 13
hook_name = f"blocks.{layer}.hook_out"
prompt = "The Eiffel Tower is located in the city of"

_, cache = model.run_with_cache(
    prompt,
    names_filter=lambda name: name == hook_name,
)
activations = cache[hook_name]
scores = ica_lens.transform(activations, layer)

print("activations:", tuple(activations.shape))
print("ICA scores: ", tuple(scores.shape))

activations: (1, 10, 2304)
ICA scores:  (1, 10, 2304)


## 3. Baseline: Logit Lens and J-Lens on the complete residual

As the baseline, read the complete final-token residual directly with Logit Lens, then transport it through J-Lens and read it again. Neither baseline readout clearly identifies the answer *France*.

In [3]:
last_token_residual = activations[0, -1]
logit_lens_residual = logit_readout(model, last_token_residual, top_k=10)
transported_residual = j_lens.transport(last_token_residual, layer)
j_lens_residual = logit_readout(model, transported_residual, top_k=10)

print("Logit Lens readout of the complete residual:")
print(logit_lens_residual.decode(model.tokenizer))
print("\nJ-Lens readout of the complete residual:")
print(j_lens_residual.decode(model.tokenizer))

Logit Lens readout of the complete residual:
['بوابة', ' town', 'Américaine', 'かわらず', ' of', 'AppCompatTheme', 'ſhip', 'Specifiche', '^(@)', 'ambassade']

J-Lens readout of the complete residual:
['########.', 'ⓧ', ' Globalization', ' cities', ' metropolis', ' Cities', ' globalization', ' urbanization', 'LookAnd', ' Metropolis']


## 4. Treatment: Logit Lens and J-Lens on ICA components

For the treatment group, rank the ICA coordinates by absolute score, then read each signed writing direction directly with Logit Lens and after transport with J-Lens. Compared with the complete-residual baseline, the components separate distinct associations: notably, *France* appears in the second strongest component's J-Lens tokens.

In [4]:
last_token_scores = scores[0, -1]
_, component_ids = last_token_scores.abs().topk(10)
for component in component_ids.tolist():
    print(f"component {component:3d}: {last_token_scores[component].item():+.4f}")

component  21: +6.4375
component 237: +6.4375
component 1159: +4.9375
component  91: +4.8438
component 1426: +4.7812
component 244: +4.4375
component 1037: -3.9062
component 565: +3.4062
component 507: +3.2344
component 1799: -3.0938


In [5]:
writing_directions = ica_lens.writing_matrices[layer][:, component_ids.cpu()].T.to(device)
signed_directions = (
    writing_directions
    * last_token_scores[component_ids].float().sign().unsqueeze(-1)
)
transported_components = j_lens.transport(signed_directions, layer)
logit_lens_components = logit_readout(model, signed_directions, top_k=5)
j_lens_components = logit_readout(model, transported_components, top_k=5)
logit_lens_tokens = logit_lens_components.decode(model.tokenizer)
j_lens_tokens = j_lens_components.decode(model.tokenizer)

for component, score, logit_tokens, j_tokens in zip(
    component_ids.tolist(),
    last_token_scores[component_ids].tolist(),
    logit_lens_tokens,
    j_lens_tokens,
):
    print(f"component {component:4d} ({score:+.4f})")
    print(f"  Logit Lens: {logit_tokens}")
    print(f"  J-Lens:     {j_tokens}")

component   21 (+6.4375)
  Logit Lens: ['.)}', '.}(', '"])\r', ' BoxDecoration', ' OF']
  J-Lens:     [' «', ' Orange', ' Oran', ' oleo', ' Os']
component  237 (+6.4375)
  Logit Lens: [' South', ' south', ' north', ' North', ' northern']
  J-Lens:     [' Europe', ' Italy', ' Germany', ' France', ' Australia']
component 1159 (+4.9375)
  Logit Lens: [' city', ' town', ' setempat', ' local', ' district']
  J-Lens:     [' town', ' municipality', ' towns', ' city', ' borough']
component   91 (+4.8438)
  Logit Lens: [' in', ',', '.', ' /', ' of']
  J-Lens:     [' is', '.', ' XNUMX', ' was', ' of']
component 1426 (+4.7812)
  Logit Lens: [' Мексичка', '省市镇', ' inaugurated', 'KommentareTeilen', 'RTDA']
  J-Lens:     [' Region', ' Province', ' Counties', ' region', ' Regions']
component  244 (+4.4375)
  Logit Lens: ['IUrlHelper', 'principalColumn', ' geslacht', ' kasarigan', ' emeritus']
  J-Lens:     [' José', ' bernama', ' André', ' Richard', ' Robert']
component 1037 (-3.9062)
  Logit Lens: [

## 5. Reconstruct the activation

A full-width ICA lens can map all component scores back into residual-stream coordinates.

In [6]:
reconstructed = ica_lens.inverse_transform(scores, layer)
error = (activations.float() - reconstructed.float()).abs()

print("reconstructed:         ", tuple(reconstructed.shape))
print("maximum absolute error:", error.max().item())
print("mean absolute error:   ", error.mean().item())

reconstructed:          (1, 10, 2304)
maximum absolute error: 2.0
mean absolute error:    0.01761145330965519
